In [0]:

from pyspark.sql.types import ArrayType, StructType, StructField, StringType
from pyspark.sql.functions import from_json, col, explode_outer,from_unixtime
from delta.tables import DeltaTable


#################################
#######---sales_orders---######
#################################


def sales_orders(df):
    sales_orders = df.select(
        "order_number",
        "customer_id",
        "customer_name",
        "number_of_line_items",
        from_unixtime(col("order_datetime")).alias("order_timestamp"))\
            .distinct()
    return sales_orders


#################################
#######---order_products---######
#################################


def ordred_products(df):
        order_product_schema = ArrayType(
            StructType([
                StructField("curr", StringType()),
                StructField("id", StringType()),
                StructField("name", StringType()),
                StructField("price", StringType()),
                StructField("promotion_info", StringType()),
                StructField("qty", StringType()),
                StructField("unit", StringType())
                ])
            )


        df_ordred_products = df.withColumn("ordered_products", from_json(col("ordered_products"),order_product_schema))\
                    .withColumn("ordered_products", explode_outer("ordered_products"))\
                    .select(
                                "customer_id",
                                "customer_name",
                                "order_number",
                                "ordered_products.id",
                                col("ordered_products.name"). alias ("product_name"),
                                "ordered_products.price",
                                "ordered_products.qty",
                                "ordered_products.unit",
                                "ordered_products.curr"
                            ).distinct()
        return df_ordred_products



#################################
#######---promotion---######
#################################


def promotions(df):
    promo_info_schema = ArrayType(
        StructType([
            StructField("promo_disc", StringType()),
            StructField("promo_id", StringType()),
            StructField("promo_item", StringType()),
            StructField("promo_qty", StringType())
        ])
    )


    df_promo = df.withColumn("promo_info", from_json(col("promo_info"), promo_info_schema))\
                  .withColumn("promo_info", explode_outer("promo_info")).filter(col("promo_info").isNotNull())\
                  .select(
                            "customer_id",
                            "customer_name",
                            "order_number",
                            col("promo_info.promo_id").alias("promo_id"),
                            col("promo_info.promo_item").alias("promo_product_id"),
                            col("promo_info.promo_disc").alias("promo_discount"),
                            col("promo_info.promo_qty").cast("int").alias("promo_quantity"))\
                  .dropDuplicates()
    return df_promo            




#################################
#######---clicked_item---######
#################################

def clicked_items(df):
        clicked_items_schema = ArrayType(
            ArrayType(StringType())
        )

        clicked_items = df.withColumn("clicked_items", from_json(col("clicked_items"), clicked_items_schema))\
                        .withColumn("clicked_items", explode_outer("clicked_items"))\
                        .select(
                                "customer_id",
                                "customer_name",
                                "order_number",
                                col("clicked_items")[0].alias("product_id"),
                                col("clicked_items")[1].alias("score")
                            ).distinct()
        return clicked_items




def scd_merge_table(spark, source_table, target_table, business_key):

    if not spark.catalog.tableExists(target_table):
        print("First Load: Creating Silver Table", target_table)
        source_table.write.format("delta").mode("overwrite").saveAsTable(target_table)
        print("Table Created")
    else:
        print("Incremental Load: Performing SCD Type 1 Merge",target_table)

        delta_table = DeltaTable.forName(spark, target_table)

        merge_condition = " AND ".join(
            [f"target.{col} = source.{col}" for col in business_key]
        )


        delta_table.alias("target").merge(source_table.alias("source"), 
                                        merge_condition
                                        ).whenMatchedUpdateAll()\
                                            .whenNotMatchedInsertAll()\
                                            .execute()
        print("Merge Successfully Completed")


#################################
#######---maincode---######
#################################


print(f"Reading source table")
bronze_table = "ecommerce_analytics.bronze.sales_orders"

df = spark.read.table(bronze_table)



#transformation
print(f"Transforming data started")
sales_orders_df = sales_orders(df)
ordered_products_df = ordred_products(df)
promotions_df = promotions(df)
clicked_items_df = clicked_items(df)



#write
scd_merge_table(spark, sales_orders_df,  "ecommerce_analytics.silver.sales_orders",["order_number", "number_of_line_items"])
scd_merge_table(spark, ordered_products_df, "ecommerce_analytics.silver.order_products",["order_number","id","price"])
scd_merge_table(spark, promotions_df, "ecommerce_analytics.silver.promotions",["order_number", "promo_id", "promo_product_id", "promo_quantity"])
scd_merge_table(spark, clicked_items_df, "ecommerce_analytics.silver.clicked_items",["order_number","product_id","score"])



print(F"Successfully wrote all tables") 

In [0]:

# step before performing SCD
'''
print(f"Transforming data - Processing Ordered Products Table")
df_ordred_products = df_ordred_products(df)
print(f"Writing data - Writing Ordered Products Table")
df_ordred_products.write.mode("overwrite").saveAsTable("ecommerce_analytics.silver.order_products")



print(f"Transforming data - Processing promotions Table")
df_promo = df_promo(df)
print(f"Writing data - Writing Ordered Promotions Table")
df_promo.write.mode("overwrite").saveAsTable("ecommerce_analytics.silver.promo_info")



print(f"Transforming data - Processing  clicked_items Table")
clicked_items = clicked_items(df)
print(f"Writing data - Writing clicked_items Table")
clicked_items.write.mode("overwrite").saveAsTable("ecommerce_analytics.silver.clicked_items")


print(f"Transforming data - Processing  sales_orders Table")
sales_orders = sales_orders(df)
print(f"Writing data - Writing sales_orders Table")
sales_orders.write.mode("overwrite").saveAsTable("ecommerce_analytics.silver.sales_orders")'''